# 03 — Regression Models

Four models estimating renewable energy support:
1. OLS baseline
2. OLS + coal × ideology interaction
3. Mixed-effects model (MLM) with state random intercepts
4. OLS on swing-state subsample

All continuous predictors are **standardized** (mean=0, SD=1) before estimation.

| | |
|---|---|
| **Inputs** | `data/processed/merged_analysis.csv` |
| **Outputs** | `output/tables/summary_table_four_models.csv`, `output/figures/marginal_effects_interaction.png` |

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_PROC = Path("../data/processed")
FIG_OUT   = Path("../output/figures")
TABLE_OUT = Path("../output/tables")
for p in [FIG_OUT, TABLE_OUT]:
    p.mkdir(parents=True, exist_ok=True)

GREEN = "#00693E"
DARK  = "#1a1a2e"

# ── Swing states (2024 election) ───────────────────────────────────────────────
SWING_STATES = [
    "Pennsylvania", "Michigan", "Wisconsin", "Arizona", "Nevada",
    "Georgia", "North Carolina", "Minnesota", "New Hampshire",
    "Virginia", "Florida", "Ohio"
]

# ── Load data ─────────────────────────────────────────────────────────────────
df = pd.read_csv(DATA_PROC / "merged_analysis.csv")
print(f"Loaded: {df.shape}  |  states: {df['state'].nunique()}")

In [ ]:
# ── Standardize all continuous predictors ────────────────────────────────────
PREDICTORS = ["ideology", "climate_concern", "education", "income", "age", "coal_share"]
PREDICTORS = [c for c in PREDICTORS if c in df.columns]

for col in PREDICTORS:
    df[f"{col}_z"] = (df[col] - df[col].mean()) / df[col].std()

print("Standardized predictors:", [f"{c}_z" for c in PREDICTORS])
print(f"\nDV (re_support) — mean: {df['re_support'].mean():.3f}  SD: {df['re_support'].std():.3f}")

In [ ]:
# ── Model 1: OLS Baseline ─────────────────────────────────────────────────────
formula_base = (
    "re_support ~ ideology_z + climate_concern_z + education_z "
    "+ income_z + age_z + coal_share_z"
)

m1 = smf.ols(formula_base, data=df).fit()
print("MODEL 1: OLS Baseline")
print(m1.summary2().tables[1].round(4))
print(f"R² = {m1.rsquared:.4f}  |  N = {int(m1.nobs)}")

In [ ]:
# ── Model 2: OLS + Interaction ────────────────────────────────────────────────
formula_int = formula_base + " + ideology_z:coal_share_z"

m2 = smf.ols(formula_int, data=df).fit()
print("MODEL 2: OLS + Coal × Ideology Interaction")
print(m2.summary2().tables[1].round(4))
print(f"R² = {m2.rsquared:.4f}  |  N = {int(m2.nobs)}")

In [ ]:
# ── Figure: Marginal effects of coal share at low/med/high ideology ───────────
coal_vals = np.linspace(df["coal_share_z"].min(), df["coal_share_z"].max(), 100)

# Predicted RE support at 3 ideology levels (−1 SD, mean, +1 SD)
fig, ax = plt.subplots(figsize=(8, 5))

ideology_levels = {
    "Liberal (−1 SD)": -1,
    "Moderate (mean)": 0,
    "Conservative (+1 SD)": 1
}
colors_ideo = [GREEN, "#7CB98F", DARK]

# Get mean values for other covariates
covariate_means = {col: 0 for col in ["climate_concern_z", "education_z", "income_z", "age_z"]}

for (label, ideo_val), color in zip(ideology_levels.items(), colors_ideo):
    pred_df = pd.DataFrame({
        "ideology_z": ideo_val,
        "coal_share_z": coal_vals,
        **covariate_means
    })
    preds = m2.predict(pred_df)
    ax.plot(coal_vals, preds, color=color, linewidth=2, label=label)

ax.set_xlabel("Coal Share (standardized)", fontsize=11)
ax.set_ylabel("Predicted RE Support", fontsize=11)
ax.set_title("Marginal Effect of Coal Share on RE Support\nby Political Ideology",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(FIG_OUT / "marginal_effects_interaction.png")
plt.show()
print("Saved: marginal_effects_interaction.png")

In [ ]:
# ── Model 3: Mixed-Effects Model (MLM) ───────────────────────────────────────
# Uses pymer4 which wraps lme4 in R via rpy2.
# If pymer4 is unavailable, use statsmodels MixedLM as fallback.

try:
    from pymer4.models import Lmer
    formula_mlm = (
        "re_support ~ ideology_z + climate_concern_z + education_z "
        "+ income_z + age_z + coal_share_z + (1 | state)"
    )
    m3 = Lmer(formula_mlm, data=df)
    m3.fit()
    print("MODEL 3: Mixed-Effects Model (pymer4/lme4)")
    print(m3.coefs.round(4))

    # ICC
    var_state   = m3.ranef_var.loc["state", "Var"]
    var_resid   = m3.ranef_var.loc["Residual", "Var"]
    icc = var_state / (var_state + var_resid)
    print(f"\nICC = {icc:.4f}  (proportion of variance due to state)")

except ImportError:
    print("pymer4 not available — using statsmodels MixedLM as fallback")
    from statsmodels.regression.mixed_linear_model import MixedLM
    exog_cols = ["ideology_z", "climate_concern_z", "education_z",
                 "income_z", "age_z", "coal_share_z"]
    exog_cols = [c for c in exog_cols if c in df.columns]
    import statsmodels.api as sm
    X = sm.add_constant(df[exog_cols])
    m3 = MixedLM(df["re_support"], X, groups=df["state"]).fit()
    print(m3.summary())

    var_state = m3.cov_re.iloc[0, 0]
    var_resid = m3.scale
    icc = var_state / (var_state + var_resid)
    print(f"\nICC = {icc:.4f}")

In [ ]:
# ── Model 4: Swing State Subgroup OLS ────────────────────────────────────────
df_swing = df[df["state"].isin(SWING_STATES)].copy()
print(f"Swing state sample: {df_swing.shape[0]} respondents, {df_swing['state'].nunique()} states")

m4 = smf.ols(formula_base, data=df_swing).fit()
print("\nMODEL 4: Swing State OLS")
print(m4.summary2().tables[1].round(4))
print(f"R² = {m4.rsquared:.4f}  |  N = {int(m4.nobs)}")

In [ ]:
# ── Export combined coefficient table ────────────────────────────────────────
def extract_coefs(model, name):
    tbl = model.summary2().tables[1][["Coef.", "Std.Err.", "P>|t|"]].copy()
    tbl.columns = [f"{name}_coef", f"{name}_se", f"{name}_p"]
    return tbl

coef_table = (
    extract_coefs(m1, "OLS")
    .join(extract_coefs(m2, "OLS_Int"), how="outer")
    .join(extract_coefs(m4, "Swing"), how="outer")
)

out_path = TABLE_OUT / "summary_table_four_models.csv"
coef_table.round(4).to_csv(out_path)
print(f"Saved: {out_path}")
coef_table.round(4)